In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession

def extrair_e_carregar_clima():
    """
    EXTRACT & LOAD (ELT): Obtém o histórico diário de clima para Bombinhas, SC
    desde 2011 até à data mais recente disponível na API de Arquivo da Open-Meteo,
    carregando os dados brutos na camada Bronze em formato Delta Lake.
    """
    # Coordenadas geográficas oficiais de Bombinhas, SC
    latitude = -27.1595
    longitude = -48.5137
    
    # Datas dinâmicas para o histórico
    data_inicial = "2011-01-01"
    
    # Margem de 5 dias para respeitar o atraso de consolidação do modelo ERA5
    data_final = (datetime.today()).strftime("%Y-%m-%d")
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": data_inicial,
        "end_date": data_final,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,wind_speed_10m_max",
        "timezone": "America/Sao_Paulo"
    }
    
    print(f"🔄 [Extract] A aceder à API Open-Meteo de {data_inicial} até {data_final}...")
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        dados_brutos = response.json()
    except Exception as e:
        raise RuntimeError(f"❌ Falha ao consumir a API Open-Meteo: {e}")
        
    daily_data = dados_brutos.get("daily", {})
    
    if not daily_data or "time" not in daily_data:
        raise RuntimeError("❌ Nenhum dado climático diário foi retornado pela API.")
        
    # 1. Criar DataFrame do Pandas com os dados da chave 'daily'
    df_pandas = pd.DataFrame(daily_data)
    
    # Renomear colunas para um padrão limpo e uniforme (snake_case) antes de gravar
    df_pandas.rename(columns={
        "time": "data",
        "temperature_2m_max": "temp_max",
        "temperature_2m_min": "temp_min",
        "precipitation_sum": "precipitacao_total",
        "rain_sum": "chuva_total",
        "wind_speed_10m_max": "vento_max"
    }, inplace=True)
    
    # 2. Converter para Spark DataFrame
    df_spark = spark.createDataFrame(df_pandas)
    
    # 3. Garantir a existência do esquema bronze
    print("📁 A garantir que o esquema 'bronze' existe...")
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    
    # 4. [Load] Gravar na tabela Bronze como Delta Lake (sobrescrevendo os dados)
    print("💾 [Load] A guardar dados climáticos na tabela bronze.clima_bombinhas_raw...")
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("bronze.clima_bombinhas_raw")
        
    print("✅ Ingestão de Clima (Open-Meteo) concluída com sucesso!")

# Executar o pipeline de clima
extrair_e_carregar_clima()

In [0]:
%sql
SELECT * FROM bronze.clima_bombinhas_raw ORDER BY data DESC LIMIT 10;